In [ ]:
!pip install torchinfo

In [ ]:
# 尝试安装较旧的版本（如0.6.x系列）
!pip install timm==0.6.13

# 或者更新到最新版本，查看是否有替代实现
!pip install --upgrade timm

  Using cached timm-0.6.13-py3-none-any.whl.metadata (38 kB)
Using cached timm-0.6.13-py3-none-any.whl (549 kB)
  Attempting uninstall: timm
    Found existing installation: timm 1.0.19
    Uninstalling timm-1.0.19:
      Successfully uninstalled timm-1.0.19


  Using cached timm-1.0.19-py3-none-any.whl.metadata (60 kB)
Using cached timm-1.0.19-py3-none-any.whl (2.5 MB)
  Attempting uninstall: timm
    Found existing installation: timm 0.6.13
    Uninstalling timm-0.6.13:
      Successfully uninstalled timm-0.6.13


In [2]:
!pip install torch torchvision medmnist numpy pyyaml pillow


In [3]:
!pip install wandb tensorboard torchinfo spikingjelly


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 9.6 MB/s eta 0:00:00


In [ ]:
import os
import time
import logging
import argparse
import yaml
import numpy as np
import random
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.backends.cudnn as cudnn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from contextlib import suppress
from collections import OrderedDict
import sys

import medmnist
from medmnist import INFO, Evaluator

# Import spikingjelly related modules
try:
    from spikingjelly.activation_based import functional, layer, neuron
    from spikingjelly.activation_based.model import train_classify
    SPIKINGJELLY_AVAILABLE = True
except ImportError:
    SPIKINGJELLY_AVAILABLE = False
    print("Warning: SpikingJelly not available. SNN functionality will be limited.")

# Define DVS dataset constant
DVS_DATASET = ["cifar10-dvs", "cifar10-dvs-tet", "dvs128"]

# Check for optional dependencies
try:
    import wandb
    has_wandb = True
except ImportError:
    has_wandb = False

try:
    from torch.cuda.amp import autocast as autocast_native, GradScaler as GradScalerNative
    has_native_amp = True
except ImportError:
    has_native_amp = False

try:
    from apex import amp
    from apex.parallel import DistributedDataParallel as DDP
    from apex.parallel import convert_syncbn_model
    has_apex = True
except ImportError:
    has_apex = False

try:
    import torchinfo
    has_torchinfo = True
except ImportError:
    has_torchinfo = False

# Create argument parser
parser = argparse.ArgumentParser(description='SNN+Transformer Training for MedMNIST')
config_parser = parser.add_argument_group('Config', description='Configuration file options')
config_parser.add_argument(
    '--config', default='', type=str, metavar='FILE',
    help='YAML config file specifying default arguments')

# Add arguments to parser
parser.add_argument(
    "--model",
    default="spikformer",
    type=str,
    metavar="MODEL",
    help="Name of model to train (default: \"spikformer\")",
)
parser.add_argument(
    "--pretrained",
    action="store_true",
    default=False,
    help="Start with pretrained version of specified network (if avail)",
)
parser.add_argument(
    "--num-classes",
    type=int,
    default=None,
    metavar="N",
    help="number of label classes (Model default if None)",
)
parser.add_argument(
    "--img-size",
    type=int,
    default=224,
    metavar="N",
    help="Image patch size (default: 224)",
)
parser.add_argument(
    "--batch-size",
    type=int,
    default=32,
    metavar="N",
    help="Input batch size for training (default: 32)",
)
parser.add_argument(
    "--workers",
    type=int,
    default=4,
    metavar="N",
    help="number of data loading workers (default: 4)",
)
parser.add_argument(
    "--validation-batch-size-multiplier",
    type=int,
    default=1,
    metavar="N",
    help="Ratio of validation batch size to training batch size (default: 1)",
)
parser.add_argument(
    "--drop",
    type=float,
    default=0.0,
    metavar="PCT",
    help="Dropout rate (default: 0.)",
)
parser.add_argument(
    "--drop-path",
    type=float,
    default=0.0, # Changed default from None to 0.0
    metavar="PCT",
    help="Drop path rate (default: None)",
)
parser.add_argument(
    "--model-ema",
    action="store_true",
    default=False,
    help="Enable tracking moving average of model weights",
)
parser.add_argument(
    "--model-ema-decay",
    type=float,
    default=0.9998,
    help="decay factor for model weights moving average (default: 0.9998)",
)
parser.add_argument(
    "--opt",
    default="adamw",
    type=str,
    metavar="OPTIMIZER",
    help='Optimizer (default: "adamw")',
)
parser.add_argument(
    "--lr",
    type=float,
    default=5e-4,
    metavar="LR",
    help="learning rate (default: 5e-4)",
)
parser.add_argument(
    "--weight-decay",
    type=float,
    default=0.05,
    help="weight decay (default: 0.05)",
)
parser.add_argument(
    "--epochs",
    type=int,
    default=100,
    metavar="N",
    help="number of epochs to train (default: 100)",
)
parser.add_argument(
    "--time-steps",
    type=int,
    default=6,
    metavar="N",
    help="Time steps for SNN (default: 6)",
)
parser.add_argument(
    "--spike-mode",
    type=str,
    default="lif",
    metavar="MODE",
    help="Spike mode for SNN (default: lif)",
)
parser.add_argument(
    "--TET",
    action="store_true",
    default=False,
    help="Use Temporal Efficient Training",
)
parser.add_argument(
    "--dataset",
    type=str,
    default="pathmnist",
    metavar="NAME",
    help="Name of MedMNIST dataset (default: pathmnist)",
)
parser.add_argument(
    "--as_rgb",
    action="store_true",
    default=False,
    help="Convert grayscale to RGB",
)
parser.add_argument(
    "--shape_transform",
    action="store_true",
    default=False,
    help="Apply shape transform for 3D datasets",
)
parser.add_argument(
    "--num_heads",
    type=int,
    default=8,
    metavar="N",
    help="Number of heads for transformer (default: 8)",
)
parser.add_argument(
    "--pooling_stat",
    type=str,
    default="avg",
    metavar="STAT",
    help="Pooling statistic for transformer (default: avg)",
)
parser.add_argument(
    "--patch_size",
    type=int,
    default=16,
    metavar="N",
    help="Patch size for transformer (default: 16)",
)
parser.add_argument(
    "--dim",
    type=int,
    default=512,
    metavar="N",
    help="Embedding dimension for transformer (default: 512)",
)
parser.add_argument(
    "--mlp_ratio",
    type=int,
    default=4,
    metavar="N",
    help="MLP ratio for transformer (default: 4)",
)
parser.add_argument(
    "--in_channels",
    type=int,
    default=3,
    metavar="N",
    help="Input channels for model (default: 3)",
)
parser.add_argument(
    "--layer",
    type=int,
    default=8,
    metavar="N",
    help="Number of layers for transformer (default: 8)",
)
parser.add_argument(
    "--seed",
    type=int,
    default=42,
    metavar="S",
    help="random seed (default: 42)",
)
parser.add_argument(
    "--amp",
    action="store_true",
    default=False,
    help="Use AMP (Automatic Mixed Precision) training",
)
parser.add_argument(
    "--output",
    default="",
    type=str,
    metavar="PATH",
    help="path to output folder (default: none, current dir)",
)
parser.add_argument(
    "--experiment",
    default="",
    type=str,
    metavar="NAME",
    help="name of train experiment, name of sub-folder for output",
)
parser.add_argument(
    "--eval-metric",
    default="top1",
    type=str,
    metavar="EVAL_METRIC",
    help='Best metric (default: "top1")',
)
parser.add_argument("--local_rank", default=0, type=int)
parser.add_argument(
    "--log-wandb",
    action="store_true",
    default=False,
    help="log training and validation metrics to wandb",
)
parser.add_argument(
    "--log-interval",
    type=int,
    default=50,
    metavar="N",
    help="how many batches to wait before logging training status (default: 50)",
)
parser.add_argument(
    "--dvs-aug",
    action="store_true",
    default=False,
    help="Enable DVS augmentation",
)
parser.add_argument(
    "--dvs-trival-aug",
    action="store_true",
    default=False,
    help="Enable DVS trivial augmentation",
)
parser.add_argument(
    "--debug",
    action="store_true",
    default=False,
    help="Enable debug logging",
)
parser.add_argument(
    "--clip-grad",
    type=float,
    default=None, # Added default None
    metavar="NORM",
    help="Clip gradient norm (default: None, no clipping)",
)
parser.add_argument(
    "--clip-mode",
    type=str,
    default="norm", # Added default "norm"
    help='Gradient clipping mode. One of ("norm", "value", "agc")',
)


_logger = logging.getLogger("train")
stream_handler = logging.StreamHandler()
format_str = "%(asctime)s %(levelname)s: %(message)s"
stream_handler.setFormatter(logging.Formatter(format_str))
_logger.addHandler(stream_handler)
_logger.propagate = False

def _parse_args():
    # Check for and remove Colab's default notebook arguments
    if '-f' in sys.argv:
        index = sys.argv.index('-f')
        # Remove '-f' and the following argument (the kernel json file path)
        sys.argv = sys.argv[:index] + sys.argv[index+2:]

    # Do we have a config file to parse?
    args_config, remaining = parser.parse_known_args()
    if args_config.config:
        with open(args_config.config, "r") as f:
            cfg = yaml.safe_load(f)
            parser.set_defaults(**cfg)

    # The main arg parser parses the rest of the args, the usual
    # defaults will have been overridden if config file specified.
    args = parser.parse_args(remaining)

    # Cache the args as a text string to save them in the output dir later
    args_text = yaml.dump(args.__dict__, default_flow_style=False)
    return args, args_text


class Transform3D:
    def __init__(self, mul=None):
        self.mul = mul

    def __call__(self, voxel):
        if self.mul == '0.5':
            voxel = voxel * 0.5
        elif self.mul == 'random':
            voxel = voxel * np.random.uniform()

        return voxel.astype(np.float32)


class AverageMeter:
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


class NativeScaler:
    def __init__(self):
        self._scaler = GradScalerNative()

    def __call__(self, loss, optimizer, clip_grad=None, clip_mode='norm', parameters=None, create_graph=False):
        self._scaler.scale(loss).backward(create_graph=create_graph)
        if clip_grad is not None:
            self._clip_grad(parameters, clip_grad, clip_mode)
        self._scaler.step(optimizer)
        self._scaler.update()

    def _clip_grad(self, parameters, clip_grad, clip_mode):
        if clip_mode == 'norm':
            torch.nn.utils.clip_grad_norm_(parameters, clip_grad)
        elif clip_mode == 'value':
            torch.nn.utils.clip_grad_value_(parameters, clip_grad)
        else:
            assert False, f"Unknown clip mode {clip_mode}"

    def state_dict(self):
        return self._scaler.state_dict()

    def load_state_dict(self, state_dict):
        self._scaler.load_state_dict(state_dict)


def distribute_bn(model, world_size, reduce=False):
    """Distribute BatchNorm stats between processes."""
    for name, param in model.named_parameters():
        if name.endswith("bn.weight") or name.endswith("bn.bias") or name.endswith("bn.running_mean") or name.endswith("bn.running_var"):
            if reduce:
                # reduce
                dist.all_reduce(param.data, op=dist.ReduceOp.SUM)
                param.data /= world_size
            else:
                # broadcast
                dist.broadcast(param.data, src=0)


def reduce_tensor(tensor, world_size):
    """Reduce tensor across all nodes."""
    rt = tensor.clone()
    dist.all_reduce(rt, op=dist.ReduceOp.SUM)
    rt /= world_size
    return rt


def dispatch_clip_grad(parameters, value, mode='norm'):
    """Dispatch gradient clipping."""
    if mode == 'norm':
        torch.nn.utils.clip_grad_norm_(parameters, value)
    elif mode == 'value':
        torch.nn.utils.clip_grad_value_(parameters, value)
    else:
        assert False, f"Unknown clip mode {mode}"


def model_parameters(model, exclude_head=False):
    """Return model parameters."""
    if exclude_head:
        return [p for n, p in model.named_parameters() if 'head' not in n]
    else:
        return model.parameters()


def setup_default_logging():
    """Setup default logging."""
    logging.basicConfig(level=logging.INFO)


def random_seed(seed, rank=0):
    """Set random seed."""
    torch.manual_seed(seed + rank)
    np.random.seed(seed + rank)
    random.seed(seed + rank)


def create_model(model_name, **kwargs):
    """Create a model based on the model name."""
    # This is a placeholder for model creation
    # In a real implementation, this would create the actual SNN+Transformer model
    if model_name == "spikformer":
        # Placeholder for Spikformer model
        from timm.models import vision_transformer
        model = vision_transformer.vit_base_patch16_224(
            num_classes=kwargs.get('num_classes', 10),
            drop_rate=kwargs.get('drop_rate', 0.0),
            drop_path_rate=kwargs.get('drop_path_rate', 0.0),
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model


def get_optimizer(model, opt='adamw', lr=1e-3, weight_decay=0.05, **kwargs):
    """Get optimizer."""
    if opt == 'adamw':
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif opt == 'sgd':
        momentum = kwargs.get('momentum', 0.9)
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unknown optimizer: {opt}")


def get_learning_rate_scheduler(optimizer, sched='cosine', epochs=100, **kwargs):
    """Get learning rate scheduler."""
    if sched == 'cosine':
        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    elif sched == 'step':
        decay_epochs = kwargs.get('decay_epochs', 30)
        decay_rate = kwargs.get('decay_rate', 0.1)
        lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=decay_epochs, gamma=decay_rate)
    else:
        lr_scheduler = None

    return lr_scheduler, epochs


class ModelEma:
    """Model Exponential Moving Average."""
    def __init__(self, model, decay=0.9999, device=''):
        self.ema_model = model
        self.decay = decay
        self.device = device

        if device:
            self.ema_model.to(device=device)

    def update(self, model):
        """Update EMA model."""
        with torch.no_grad():
            for ema_v, model_v in zip(self.ema_model.state_dict().values(), model.state_dict().values()):
                if self.device:
                    ema_v.copy_(ema_v * self.decay + model_v.to(self.device) * (1.0 - self.decay))
                else:
                    ema_v.copy_(ema_v * self.decay + model_v * (1.0 - self.decay))

    def state_dict(self):
        """Return state dict."""
        return self.ema_model.state_dict()

    def load_state_dict(self, state_dict):
        """Load state dict."""
        self.ema_model.load_state_dict(state_dict)


class CheckpointSaver:
    """Checkpoint Saver."""
    def __init__(self, model, optimizer, args=None, model_ema=None, amp_scaler=None,
                 checkpoint_dir='./checkpoints', recovery_dir='./recoveries', max_history=10):
        self.model = model
        self.optimizer = optimizer
        self.args = args
        self.model_ema = model_ema
        self.amp_scaler = amp_scaler
        self.checkpoint_dir = checkpoint_dir
        self.recovery_dir = recovery_dir
        self.max_history = max_history

        os.makedirs(checkpoint_dir, exist_ok=True)
        os.makedirs(recovery_dir, exist_ok=True)

        self.best_metric = None
        self.best_epoch = None

    def save_checkpoint(self, epoch, metric=None):
        """Save checkpoint."""
        if metric is not None and (self.best_metric is None or metric > self.best_metric):
            self.best_metric = metric
            self.best_epoch = epoch

            # Save best checkpoint
            checkpoint = {
                'epoch': epoch,
                'model': self.model.state_dict(),
                'optimizer': self.optimizer.state_dict(),
                'best_metric': metric,
            }

            if self.model_ema is not None:
                checkpoint['model_ema'] = self.model_ema.state_dict()

            if self.amp_scaler is not None:
                checkpoint['scaler'] = self.amp_scaler.state_dict()

            torch.save(checkpoint, os.path.join(self.checkpoint_dir, 'model_best.pth.tar'))

        return self.best_metric, self.best_epoch

    def save_recovery(self, epoch, batch_idx=None):
        """Save recovery checkpoint."""
        if batch_idx is not None:
            recovery_name = f'recovery_epoch_{epoch}_batch_{batch_idx}.pth.tar'
        else:
            recovery_name = f'recovery_epoch_{epoch}.pth.tar'

        checkpoint = {
            'epoch': epoch,
            'model': self.model.state_dict(),
            'optimizer': self.optimizer.state_dict(),
        }

        if self.model_ema is not None:
            checkpoint['model_ema'] = self.model_ema.state_dict()

        if self.amp_scaler is not None:
            checkpoint['scaler'] = self.amp_scaler.state_dict()

        torch.save(checkpoint, os.path.join(self.recovery_dir, recovery_name))


def update_summary(epoch, train_metrics, eval_metrics, filename, write_header=False, log_wandb=False):
    """Update summary CSV."""
    if not os.path.exists(filename):
        write_header = True

    with open(filename, mode='a' if not write_header else 'w') as f:
        if write_header:
            f.write('epoch,train_loss,eval_loss,eval_top1,eval_top5\n')

        f.write(f'{epoch},{train_metrics["loss"]:.4f},{eval_metrics["loss"]:.4f},'
                f'{eval_metrics["top1"]:.4f},{eval_metrics["top5"]:.4f}\n')

    if log_wandb:
        wandb.log({
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'eval_loss': eval_metrics['loss'],
            'eval_top1': eval_metrics['top1'],
            'eval_top5': eval_metrics['top5'],
        })


def validate(model, loader, loss_fn, args, amp_autocast=suppress, log_suffix=''):
    """Validate the model."""
    batch_time_m = AverageMeter()
    losses_m = AverageMeter()
    top1_m = AverageMeter()
    top5_m = AverageMeter()

    model.eval()

    end = time.time()
    last_idx = len(loader) - 1

    with torch.no_grad():
        for batch_idx, (input, target) in enumerate(loader):
            input = input.to(args.device)
            target = target.to(args.device)

            # Ensure target is long and 1D for CrossEntropyLoss
            target = target.long().squeeze()
            if target.dim() > 1:
                 # If after squeezing, target is still not 1D, something is wrong with the data
                 _logger.error(f"Unexpected target shape after squeeze: {target.shape}")
                 continue


            with amp_autocast():
                output = model(input)
                if isinstance(output, tuple):
                    output = output[0]

                loss = loss_fn(output, target)

            if output.dim() > 1:
                # For classification tasks
                acc1, acc5 = accuracy(output, target, topk=(1, 5))
                top1_m.update(acc1.item(), input.size(0))
                top5_m.update(acc5.item(), output.size(0))

            losses_m.update(loss.item(), input.size(0))

            batch_time_m.update(time.time() - end)
            end = time.time()

            if batch_idx % args.log_interval == 0:
                _logger.info(
                    'Test: [{0:>4d}/{1}]  '
                    'Time: {batch_time.val:.3f} ({batch_time.avg:.3f})  '
                    'Loss: {loss.val:>7.4f} ({loss.avg:>6.4f})  '
                    'Acc@1: {top1.val:>7.4f} ({top1.avg:>7.4f})  '
                    'Acc@5: {top5.val:>7.4f} ({top5.avg:>7.4f})'.format(
                        batch_idx, last_idx, batch_time=batch_time_m,
                        loss=losses_m, top1=top1_m, top5=top5_m)
                )

    metrics = OrderedDict([('loss', losses_m.avg)])

    if output.dim() > 1:
        metrics['top1'] = top1_m.avg
        metrics['top5'] = top5_m.avg

    return metrics


def accuracy(output, target, topk=(1,)):
    """Computes the accuracy over the k top predictions for the specified values of k."""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res


class Mixup:
    """Mixup/Cutmix that applies different params to each element or whole batch."""
    def __init__(self, mixup_alpha=1.0, cutmix_alpha=0.0, cutmix_minmax=None, prob=1.0,
                 switch_prob=0.5, mode='batch', correct_lam=True, label_smoothing=0.1,
                 num_classes=1000):
        self.mixup_alpha = mixup_alpha
        self.cutmix_alpha = cutmix_alpha
        self.cutmix_minmax = cutmix_minmax
        self.mixup_enabled = True  # set to false to disable mixup
        self.prob = prob
        self.switch_prob = switch_prob
        self.label_smoothing = label_smoothing
        self.num_classes = num_classes
        self.mode = mode  # 'batch', 'pair', 'elem'
        self.correct_lam = correct_lam  # correct lambda based on clipped area

    def _params_per_elem(self, batch_size):
        if self.mode == 'elem':
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha, batch_size)
            use_cutmix = np.random.rand(batch_size) < self.switch_prob
        else:
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
            use_cutmix = np.random.rand() < self.switch_prob
            lam = np.ones(batch_size) * lam
        return lam, use_cutmix

    def _mix_elem(self, x, lam):
        x_flipped = x.flip(0)  # flip batch
        return x * lam.reshape(-1, 1, 1, 1) + x_flipped * (1 - lam.reshape(-1, 1, 1, 1))

    def __call__(self, x, target):
        if not self.mixup_enabled:
            return x, target

        batch_size = x.size(0)
        lam, use_cutmix = self._params_per_elem(batch_size)

        if np.any(use_cutmix):
            # Apply cutmix
            if self.mode == 'elem':
                for i in range(batch_size):
                    if use_cutmix[i]:
                        x[i], lam[i] = self._cutmix(x[i], lam[i])
            else:
                x, lam = self._cutmix_batch(x, lam)

        if not np.all(use_cutmix):
            # Apply mixup to non-cutmix elements
            if self.mode == 'elem':
                for i in range(batch_size):
                    if not use_cutmix[i]:
                        x[i] = self._mix_elem(x[i:i+1], lam[i:i+1])[0]
            else:
                x = self._mix_elem(x, lam)

        target = mixup_target(target, self.num_classes, lam, self.label_smoothing)
        return x, target

    def _cutmix_batch(self, x, lam):
        """Cutmix for a batch of samples."""
        bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
        x[:, :, bbx1:bbx2, bby1:bby2] = x.flip(0)[:, :, bbx1:bbx2, bby1:bby2]

        # adjust lambda to exactly match pixel ratio
        lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-2] * x.size()[-1]))
        return x, lam

    def _cutmix(self, x, lam):
        """Cutmix for a single sample."""
        bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
        x[:, bbx1:bbx2, bby1:bby2] = x.flip(0)[:, bbx1:bbx2, bby1:bby2]

        # adjust lambda to exactly match pixel ratio
        lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-2] * x.size()[-1]))
        return x, lam


def rand_bbox(size, lam):
    """Generate random bounding box for cutmix."""
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = np.int(W * cut_rat)
    cut_h = np.int(H * cut_rat)

    # uniform
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2


def mixup_target(target, num_classes, lam=1.0, smoothing=0.0):
    """Create mixed up target."""
    off_value = smoothing / num_classes
    on_value = 1. - smoothing + off_value

    target1 = one_hot(target, num_classes, on_value=on_value, off_value=off_value)
    target2 = one_hot(target.flip(0), num_classes, on_value=on_value, off_value=off_value)

    return target1 * lam + target2 * (1. - lam)


def one_hot(index, classes, on_value=1., off_value=0.):
    """Convert index to one-hot encoding."""
    index = index.long()
    if index.dim() == 1:
        index = index.unsqueeze(1)

    one_hot = torch.zeros(index.size(0), classes, dtype=torch.float, device=index.device)
    one_hot.scatter_(1, index, on_value)

    return one_hot - off_value


class DVSAug:
    """DVS augmentation."""
    def __call__(self, x):
        # Placeholder for DVS augmentation
        return x


class DVSTrivalAug:
    """DVS trivial augmentation."""
    def __call__(self, x):
        # Placeholder for DVS trivial augmentation
        return x


class Criterion:
    """Criterion class with TET loss support."""
    def TET_loss(self, output, target, loss_fn, means=0.5, lamb=0.1):
        """Temporal Efficient Training loss."""
        # Placeholder for TET loss
        return loss_fn(output, target)


criterion = Criterion()


def main():
    setup_default_logging()
    args, args_text = _parse_args()

    # Set logging level based on debug flag
    if args.debug:
        _logger.setLevel(logging.DEBUG)
        _logger.debug("Debug mode enabled.")
    else:
        _logger.setLevel(logging.INFO)

    # Ensure clip_mode is a string and clip_grad has a default if not set
    if not isinstance(args.clip_mode, str):
        args.clip_mode = 'norm' # Default to 'norm' if not a string
        _logger.warning(f"args.clip_mode was not a string, defaulting to '{args.clip_mode}'")

    if args.clip_grad is None:
         args.clip_grad = 0.0 # Default to 0.0 (no clipping) if None
         _logger.info("args.clip_grad was None, defaulting to 0.0 (no clipping)")


    if args.log_wandb:
        if has_wandb:
            wandb.init(project=args.experiment, config=args)
        else:
            _logger.warning(
                "You've requested to log metrics to wandb but package not found. "
                "Metrics not being logged to wandb, try `pip install wandb`"
            )

    args.prefetcher = True  # Default to True
    args.distributed = False
    if "WORLD_SIZE" in os.environ:
        args.distributed = int(os.environ["WORLD_SIZE"]) > 1
    args.device = "cuda:0" if torch.cuda.is_available() else "cpu"
    args.world_size = 1
    args.rank = 0  # global rank

    if args.distributed:
        args.device = "cuda:%d" % args.local_rank
        torch.cuda.set_device(args.local_rank)
        torch.distributed.init_process_group(backend="nccl", init_method="env://")
        args.world_size = torch.distributed.get_world_size()
        args.rank = torch.distributed.get_rank()
        _logger.info(
            "Training in distributed mode with multiple processes, 1 GPU per process. Process %d, total %d."
            % (args.rank, args.world_size)
        )
    else:
        _logger.info("Training with a single process on %s." % args.device)

    assert args.rank >= 0

    # resolve AMP arguments based on PyTorch / Apex availability
    use_amp = None
    if args.amp:
        # `--amp` chooses native amp before apex (APEX ver not actively maintained)
        if has_native_amp:
            args.native_amp = True
        elif has_apex:
            args.apex_amp = True
    if hasattr(args, 'apex_amp') and args.apex_amp and has_apex:
        use_amp = "apex"
    elif hasattr(args, 'native_amp') and args.native_amp and has_native_amp:
        use_amp = "native"
    elif (hasattr(args, 'apex_amp') and args.apex_amp) or (hasattr(args, 'native_amp') and args.native_amp):
        _logger.warning(
            "Neither APEX or native Torch AMP is available, using float32. "
            "Install NVIDA apex or upgrade to PyTorch 1.6"
        )

    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True

    os.environ["PYTHONHASHSEED"] = str(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(args.seed)
        torch.cuda.manual_seed_all(args.seed)
    random_seed(args.seed, args.rank)

    args.dvs_mode = False
    if args.dataset in DVS_DATASET:
        args.dvs_mode = True

    download = True

    # Get dataset information
    data_flag = args.dataset
    if data_flag not in INFO:
        raise ValueError(f"Dataset {data_flag} not found in MedMNIST")

    info = INFO[data_flag]
    task = info['task']
    n_channels = info['n_channels']
    n_classes = len(info['label'])

    # Update args with dataset information
    if args.num_classes is None:
        args.num_classes = n_classes
    if args.in_channels is None:
        args.in_channels = n_channels

    # Set up transforms
    if '3d' in data_flag:
        shape_transform = args.shape_transform
        train_transform = Transform3D(mul='random') if shape_transform else Transform3D()
        test_transform = Transform3D(mul='0.5') if shape_transform else Transform3D()
        as_rgb = args.as_rgb
    else:
        train_transform = transforms.Compose([
            transforms.Resize(224),
            transforms.Lambda(lambda image: image.convert('RGB')),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[.5], std=[.5])
        ])
        test_transform = transforms.Compose([
            transforms.Resize(224),
            transforms.Lambda(lambda image: image.convert('RGB')),
            transforms.ToTensor(),
            transforms.Normalize(mean=[.5], std=[.5])
        ])

    # Load dataset
    DataClass = getattr(medmnist, info['python_class'])

    # Download and load the data
    train_dataset = DataClass(split='train', transform=train_transform, download=download, as_rgb=as_rgb if '3d' in data_flag else False)
    test_dataset = DataClass(split='test', transform=test_transform, download=download, as_rgb=as_rgb if '3d' in data_flag else False)

    # Create data loaders
    if args.distributed:
        train_sampler = DistributedSampler(train_dataset, num_replicas=args.world_size, rank=args.rank, shuffle=True)
        test_sampler = DistributedSampler(test_dataset, num_replicas=args.world_size, rank=args.rank, shuffle=False)
    else:
        train_sampler = None
        test_sampler = None

    loader_train = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=(train_sampler is None),
        sampler=train_sampler,
        num_workers=args.workers,
        pin_memory=True if torch.cuda.is_available() else False,
        drop_last=True,
    )

    loader_eval = DataLoader(
        test_dataset,
        batch_size=args.validation_batch_size_multiplier * args.batch_size,
        shuffle=False,
        sampler=test_sampler,
        num_workers=args.workers,
        pin_memory=True if torch.cuda.is_available() else False,
        drop_last=False,
    )

    # Create model
    model = create_model(
        args.model,
        T=args.time_steps,
        pretrained=args.pretrained,
        drop_rate=args.drop,
        drop_path_rate=args.drop_path,
        num_heads=args.num_heads,
        num_classes=args.num_classes,
        pooling_stat=args.pooling_stat,
        img_size_h=args.img_size,
        img_size_w=args.img_size,
        patch_size=args.patch_size,
        embed_dims=args.dim,
        mlp_ratios=args.mlp_ratio,
        in_channels=args.in_channels,
        qkv_bias=False,
        depths=args.layer,
        sr_ratios=1,
        spike_mode=args.spike_mode,
        dvs_mode=args.dvs_mode,
        TET=args.TET,
    )

    if args.local_rank == 0:
        _logger.info(f"Creating model {args.model}")
        if has_torchinfo:
            try:
                # Adjust input size for 3D data if needed
                if '3d' in data_flag:
                     _logger.info(
                         str(
                             torchinfo.summary(
                                 model, (2, args.in_channels, args.img_size, args.img_size, args.img_size)
                             )
                         )
                     )
                else:
                    _logger.info(
                        str(
                            torchinfo.summary(
                                model, (2, args.in_channels, args.img_size, args.img_size)
                            )
                        )
                    )
            except Exception as e:
                _logger.warning(f"Failed to run torchinfo.summary: {e}")

    # Move model to device
    model = model.to(args.device)

    # Setup distributed training
    if args.distributed:
        if hasattr(args, 'apex_amp') and args.apex_amp and has_apex:
            model = convert_syncbn_model(model)
        model = DDP(model, device_ids=[args.local_rank], output_device=args.local_rank)

    # Setup optimizer
    optimizer = get_optimizer(
        model,
        opt=args.opt,
        lr=args.lr,
        weight_decay=args.weight_decay,
    )

    # Setup learning rate scheduler
    lr_scheduler, num_epochs = get_learning_rate_scheduler(
        optimizer,
        sched='cosine',
        epochs=args.epochs,
    )

    # Setup loss function
    # Import SoftTargetCrossEntropy here to avoid NameError
    class SoftTargetCrossEntropy(nn.Module):
        def __init__(self):
            super(SoftTargetCrossEntropy, self).__init__()

        def forward(self, x, target):
            loss = torch.sum(-target * F.log_softmax(x, dim=-1), dim=-1)
            return loss.mean()

    if hasattr(args, 'mixup') and (args.mixup > 0. or (hasattr(args, 'cutmix') and args.cutmix > 0.)):
         train_loss_fn = SoftTargetCrossEntropy().to(args.device) # Use SoftTargetCrossEntropy for Mixup/Cutmix
    else:
        train_loss_fn = nn.CrossEntropyLoss().to(args.device)

    validate_loss_fn = nn.CrossEntropyLoss().to(args.device)

    # Setup mixup/cutmix
    mixup_fn = None
    if hasattr(args, 'mixup') and (args.mixup > 0. or (hasattr(args, 'cutmix') and args.cutmix > 0.)):
        mixup_fn = Mixup(
            mixup_alpha=args.mixup if hasattr(args, 'mixup') else 1.0,
            cutmix_alpha=args.cutmix if hasattr(args, 'cutmix') else 0.0,
            prob=args.mixup_prob if hasattr(args, 'mixup_prob') else 1.0,
            switch_prob=args.mixup_switch_prob if hasattr(args, 'mixup_switch_prob') else 0.5,
            mode=args.mixup_mode if hasattr(args, 'mixup_mode') else 'batch',
            label_smoothing=args.smoothing if hasattr(args, 'smoothing') else 0.1,
            num_classes=args.num_classes
        )

    # Setup EMA
    model_ema = None
    if hasattr(args, 'model_ema') and args.model_ema:
        model_ema = ModelEma(
            model,
            decay=args.model_ema_decay if hasattr(args, 'model_ema_decay') else 0.9998,
            device='cpu' if (hasattr(args, 'model_ema_force_cpu') and args.model_ema_force_cpu) else '',
        )

    # Setup AMP
    amp_autocast = suppress  # do nothing
    loss_scaler = None
    if use_amp == 'native':
        amp_autocast = autocast_native
        loss_scaler = NativeScaler()
        if args.local_rank == 0:
            _logger.info('Using native Torch AMP. Training in mixed precision.')
    elif use_amp == 'apex':
        if args.local_rank == 0:
            _logger.info('Using NVIDIA APEX AMP. Training in mixed precision.')
        model, optimizer = amp.initialize(model, optimizer, opt_level="O1")
        loss_scaler = ApexScaler()
    else:
        if args.local_rank == 0:
            _logger.info('AMP not enabled. Training in float32.')


    # Setup checkpoint saver
    output_dir = None
    if args.output:
        output_dir = args.output
        os.makedirs(output_dir, exist_ok=True)

    eval_metric = args.eval_metric
    best_metric = None
    best_epoch = None

    saver = None
    if output_dir:
        saver = CheckpointSaver(
            model=model,
            optimizer=optimizer,
            args=args,
            model_ema=model_ema,
            amp_scaler=loss_scaler,
            checkpoint_dir=output_dir,
            recovery_dir=output_dir,
            max_history=10
        )

    # Setup DVS augmentation
    train_dvs_aug = None
    train_dvs_trival_aug = None
    if hasattr(args, 'dvs_aug') and args.dvs_aug:
        train_dvs_aug = DVSAug()
    if hasattr(args, 'dvs_trival_aug') and args.dvs_trival_aug:
        train_dvs_trival_aug = DVSTrivalAug()

    # Setup start epoch
    start_epoch = 0
    if hasattr(args, 'start_epoch') and args.start_epoch is not None:
        start_epoch = args.start_epoch

    # Main training loop
    try:
        for epoch in range(start_epoch, num_epochs):
            if args.distributed and hasattr(loader_train.sampler, "set_epoch"):
                loader_train.sampler.set_epoch(epoch)

            train_metrics = train_one_epoch(
                epoch,
                model,
                loader_train,
                optimizer,
                train_loss_fn,
                args,
                lr_scheduler=lr_scheduler,
                saver=saver,
                output_dir=output_dir,
                amp_autocast=amp_autocast,
                loss_scaler=loss_scaler,
                model_ema=model_ema,
                mixup_fn=mixup_fn,
                dvs_aug=train_dvs_aug,
                dvs_trival_aug=train_dvs_trival_aug,
            )

            if args.distributed and hasattr(args, 'dist_bn') and args.dist_bn in ("broadcast", "reduce"):
                if args.local_rank == 0:
                    _logger.info("Distributing BatchNorm running means and vars")
                distribute_bn(model, args.world_size, args.dist_bn == "reduce")

            eval_metrics = validate(
                model, loader_eval, validate_loss_fn, args, amp_autocast=amp_autocast
            )

            if model_ema is not None and (not hasattr(args, 'model_ema_force_cpu') or not args.model_ema_force_cpu):
                if args.distributed and hasattr(args, 'dist_bn') and args.dist_bn in ("broadcast", "reduce"):
                    distribute_bn(model_ema, args.world_size, args.dist_bn == "reduce")
                ema_eval_metrics = validate(
                    model_ema.module if args.distributed else model_ema,
                    loader_eval,
                    validate_loss_fn,
                    args,
                    amp_autocast=amp_autocast,
                    log_suffix=" (EMA)",
                )
                eval_metrics = ema_eval_metrics

            if lr_scheduler is not None:
                # step LR for next epoch
                lr_scheduler.step(epoch + 1)

            if output_dir is not None:
                update_summary(
                    epoch,
                    train_metrics,
                    eval_metrics,
                    os.path.join(output_dir, "summary.csv"),
                    write_header=best_metric is None,
                    log_wandb=args.log_wandb and has_wandb,
                )

            if saver is not None:
                # save proper checkpoint with eval metric
                save_metric = eval_metrics[eval_metric]
                best_metric, best_epoch = saver.save_checkpoint(
                    epoch, metric=save_metric
                )
                _logger.info(
                    "*** Best metric: {0} (epoch {1})".format(best_metric, best_epoch)
                )

    except KeyboardInterrupt:
        pass
    if best_metric is not None:
        _logger.info("*** Best metric: {0} (epoch {1})".format(best_metric, best_epoch))


def train_one_epoch(
    epoch,
    model,
    loader,
    optimizer,
    loss_fn,
    args,
    lr_scheduler=None,
    saver=None,
    output_dir=None,
    amp_autocast=suppress,
    loss_scaler=None,
    model_ema=None,
    mixup_fn=None,
    dvs_aug=None,
    dvs_trival_aug=None,
):
    if hasattr(args, 'mixup_off_epoch') and args.mixup_off_epoch and epoch >= args.mixup_off_epoch:
        if hasattr(args, 'prefetcher') and args.prefetcher:
            if hasattr(loader, "mixup_enabled"):
                loader.mixup_enabled = False
        elif mixup_fn is not None:
            mixup_fn.mixup_enabled = False

    sample_number = 0
    start_time = time.time()

    second_order = hasattr(optimizer, "is_second_order") and optimizer.is_second_order
    batch_time_m = AverageMeter()
    data_time_m = AverageMeter()
    losses_m = AverageMeter()

    model.train()

    # Reset SNN state if using SpikingJelly
    if SPIKINGJELLY_AVAILABLE:
        functional.reset_net(model)

    end = time.time()
    last_idx = len(loader) - 1
    num_updates = epoch * len(loader)
    device = next(model.parameters()).device  # 获取模型参数所在的设备

    for batch_idx, (input, target) in enumerate(loader):
        last_batch = batch_idx == last_idx
        data_time_m.update(time.time() - end)
        input = input.to(device=device, dtype=torch.float32)
        target = target.to(device=device, dtype=torch.long) # Ensure target is long after loading

        if (hasattr(args, 'prefetcher') and not args.prefetcher) or args.dataset in DVS_DATASET:
            if hasattr(args, 'amp') and args.amp and not isinstance(input, torch.cuda.HalfTensor):
                input = input.half()
            if dvs_aug is not None:
                input = dvs_aug(input)
            if dvs_trival_aug is not None:
                output = []
                for i in range(input.shape[0]):
                    output.append(dvs_trival_aug(input[i]))
                input = torch.stack(output)
                del output


        if mixup_fn is not None:
            input, target = mixup_fn(input, target)

        # Ensure target is in the correct shape for SoftTargetCrossEntropy if mixup is used
        # If not using mixup, target should be (batch_size,) with class indices, which long() handles
        if mixup_fn is None:
             # If not using mixup, ensure target is 1D for CrossEntropyLoss
             target = target.squeeze()


        if hasattr(args, 'channels_last') and args.channels_last:
            input = input.contiguous(memory_format=torch.channels_last)

        with amp_autocast():
            output = model(input)
            if isinstance(output, tuple):
                output = output[0]
            if hasattr(args, 'TET') and args.TET:
                loss = criterion.TET_loss(
                    output, target, loss_fn, means=args.TET_means if hasattr(args, 'TET_means') else 0.5, lamb=args.TET_lamb if hasattr(args, 'TET_lamb') else 0.1
                )
            else:
                loss = loss_fn(output, target)

        sample_number += input.shape[0]
        if not args.distributed:
            losses_m.update(loss.item(), input.size(0))

        optimizer.zero_grad()
        if loss_scaler is not None:
            loss_scaler(
                loss,
                optimizer,
                clip_grad=args.clip_grad,
                clip_mode=args.clip_mode,
                parameters=model_parameters(
                    model, exclude_head=hasattr(args, 'clip_mode') and "agc" in args.clip_mode
                ),
                create_graph=second_order,
            )
        else:
            loss.backward(create_graph=second_order)
            if args.clip_grad is not None:
                dispatch_clip_grad(
                    model_parameters(model, exclude_head=hasattr(args, 'clip_mode') and "agc" in args.clip_mode),
                    value=args.clip_grad,
                    mode=args.clip_mode,
                )
            optimizer.step()

        # Reset SNN state if using SpikingJelly
        if SPIKINGJELLY_AVAILABLE:
            functional.reset_net(model)
            if model_ema is not None:
                functional.reset_net(model_ema)

        if model_ema is not None:
            model_ema.update(model)

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        num_updates += 1
        batch_time_m.update(time.time() - end)
        if last_batch or batch_idx % args.log_interval == 0:
            lrl = [param_group["lr"] for param_group in optimizer.param_groups]
            lr = sum(lrl) / len(lrl)

            if args.distributed:
                reduced_loss = reduce_tensor(loss.data, args.world_size)
                losses_m.update(reduced_loss.item(), input.size(0))

            if args.local_rank == 0:
                _logger.info(
                    "Train: {} [{:>4d}/{} ({:>3.0f}%)]  "
                    "Loss: {loss.val:>9.6f} ({loss.avg:>6.4f})  "
                    "Time: {batch_time.val:.3f}s, {rate:>7.2f}/s  "
                    "({batch_time.avg:.3f}s, {rate_avg:>7.2f}/s)  "
                    "LR: {lr:.3e}  "
                    "Data: {data_time.val:.3f} ({data_time.avg:.3f})".format(
                        epoch,
                        batch_idx,
                        len(loader),
                        100.0 * batch_idx / last_idx,
                        loss=losses_m,
                        batch_time=batch_time_m,
                        rate=input.size(0) * args.world_size / batch_time_m.val,
                        rate_avg=input.size(0) * args.world_size / batch_time_m.avg,
                        lr=lr,
                        data_time=data_time_m,
                    )
                )

                if hasattr(args, 'save_images') and args.save_images and output_dir:
                    torchvision.utils.save_image(
                        input,
                        os.path.join(output_dir, "train-batch-%d.jpg" % batch_idx),
                        padding=0,
                        normalize=True,
                    )

        if (
            saver is not None
            and hasattr(args, 'recovery_interval') and args.recovery_interval
            and (last_batch or (batch_idx + 1) % args.recovery_interval == 0)
        ):
            saver.save_recovery(epoch, batch_idx=batch_idx)

        if lr_scheduler is not None and hasattr(lr_scheduler, 'step_update'):
            lr_scheduler.step_update(num_updates=num_updates, metric=losses_m.avg)

        end = time.time()
        # end for

    if hasattr(optimizer, "sync_lookahead"):
        optimizer.sync_lookahead()
    if args.local_rank == 0:
        _logger.info(f"samples / s = {sample_number / (time.time() - start_time): .3f}")
    return OrderedDict([("loss", losses_m.avg)])


if __name__ == "__main__":
    main()

2025-09-06 07:33:19,347 INFO: args.clip_grad was None, defaulting to 0.0 (no clipping)
2025-09-06 07:33:19,403 INFO: Training with a single process on cuda:0.
100%|██████████| 206M/206M [00:35<00:00, 5.81MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
2025-09-06 07:34:01,233 INFO: Creating model spikformer
2025-09-06 07:34:02,707 INFO: ==========================================================================================
Layer (type:depth-idx)                   Output Shape              Param #
VisionTransformer                        [2, 9]                    152,064
├─PatchE